In [68]:
from __future__ import annotations

import operator
from typing import TypedDict, List, Annotated

from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
import os
from dotenv import load_dotenv

from langchain_mistralai import ChatMistralAI
from langchain_core.messages import SystemMessage, HumanMessage

In [69]:
load_dotenv()

True

In [70]:

class Task(BaseModel):
    id: int
    title: str
    brief: str = Field(..., description="What to cover")

In [71]:
class Plan(BaseModel):
    blog_title: str
    tasks: List[Task]

In [72]:
class State(TypedDict):
    topic: str
    plan: Plan
    # reducer: results from workers get concatenated automatically
    sections: Annotated[List[str], operator.add]
    final: str

In [73]:
llm = ChatMistralAI(
    model="mistral-small-latest",
    mistral_api_key=os.getenv("MISTRAL_API_KEY"),
    temperature=0.3
)

In [74]:
def orchestrator(state: State) -> dict:

    plan = llm.with_structured_output(Plan).invoke(
        [
            SystemMessage(
                content=(
                    "Create a blog plan with 5-7 sections on the following topic."
                )
            ),
            HumanMessage(content=f"Topic: {state['topic']}"),
        ]
    )
    return {"plan": plan}

In [75]:
def fanout(state: State):
    return [Send("worker", {"task": task, "topic": state["topic"], "plan": state["plan"]})
            for task in state["plan"].tasks]

In [76]:
def collect(state: State):
    print("Collected sections:", len(state["sections"]))
    return {}

In [77]:
def worker(payload: dict) -> dict:

    task = payload["task"]

    section_md = llm.invoke(
        [
            SystemMessage(
                content="Write one clean Markdown section."
            ),
            HumanMessage(
                content=f"""
Blog: {payload['plan'].blog_title}

Topic: {payload['topic']}

Section:
{task.title}

Brief:
{task.brief}

Return only markdown and only string.
"""
            )
        ]
    ).content.strip()

    print(section_md)

    return {
        "sections": [section_md]
    }

In [78]:
from pathlib import Path

def reducer(state: State) -> dict:
    
    title = state["plan"].blog_title
    body = "\n\n".join(state["sections"]).strip()

    final_md = f"# {title}\n\n{body}\n"

    # ---- save to file ----
    filename = title.lower().replace(" ", "_") + ".md"
    output_path = Path(filename)
    output_path.write_text(final_md, encoding="utf-8")

    return {"final": final_md}

In [79]:
g = StateGraph(State)

g.add_node("orchestrator", orchestrator)
g.add_node("worker", worker)
g.add_node("collect", collect)
g.add_node("reducer", reducer)

g.add_edge(START, "orchestrator")

g.add_conditional_edges(
    "orchestrator",
    fanout,
    ["worker"]
)

g.add_edge("worker", "collect")
g.add_edge("collect", "reducer")

g.add_edge("reducer", END)

app = g.compile()

In [80]:
out = app.invoke({"topic": "Write a blog on Linear Regression","sections": [], "final": ""})
print(out["final"])

```markdown
## Evaluating Model Performance: Metrics That Matter

Evaluating the performance of a linear regression model is crucial to understanding how well it predicts outcomes. Several key metrics help quantify the model's accuracy and effectiveness. Below, we explore the most important ones:

### **1. Mean Absolute Error (MAE)**
**Definition:** MAE measures the average absolute difference between predicted and actual values.
**Formula:**
\[
MAE = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|
\]
**Interpretation:**
- Lower MAE indicates better model performance.
- Example: If MAE = 2.5, the model's predictions are, on average, off by 2.5 units.

### **2. Mean Squared Error (MSE)**
**Definition:** MSE calculates the average squared difference between predicted and actual values.
**Formula:**
\[
MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2
\]
**Interpretation:**
- Penalizes larger errors more heavily due to squaring.
- Example: If MSE = 10, the model's errors are squared and a